In [ ]:
!pip install -q --upgrade transformers==4.53.2 --no-warn-conflicts
!pip install -q IndicTransToolkit

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor
import pandas as pd
import glob
from datasets import load_dataset
from concurrent.futures import ThreadPoolExecutor

In [ ]:
MODEL_NAME = "ai4bharat/indictrans2-en-indic-1B"
SOURCE_LANGUAGE="eng_Latn"
TARGET_LANGUAGE = "npi_Deva"
BATCH_SIZE=128
NUM_BEAMS = 4               # Using beam search helps find a logical path out of loops
MAX_NEW_TOKENS=128              # Prevents runaway generation
NO_REPEAT_NGRAM_SIZE=2      # Prevents repeating any 3-word sequence
REPETITION_PENALTY=1.2     # Specifically penalizes repeating tokens
EARLY_STOPPING = True
SAVE_THRESHOLD = 5120  # Save every 5120 items
LOCAL_DIR = "/content/my_data/CC3M_Captions"
SOURCE_DIR = "/content/drive/My Drive/CC3M_Captions"
TARGET_DIR = "/content/drive/MyDrive/Nep_CLIP_Datasets/Raw_Training_Data"


In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.manual_seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p /content/my_data
!cp -r "$SOURCE_DIR" /content/my_data/

In [ ]:
if not os.path.exists(TARGET_DIR):
  os.makedirs(TARGET_DIR, exist_ok=True)
  print(f"🆕 Folder created: {TARGET_DIR}")
else:
  if os.path.isdir(TARGET_DIR):
    print(f"✅ Folder already exists: {TARGET_DIR}")
  else:
    print(f"⚠️ Warning: A FILE exists at {TARGET_DIR} with this name!")

In [ ]:
def get_parquet_files(folder_path):
  """
  Takes a folder path and returns a sorted list of all .parquet files.
  """
  # 1. Normalize path and ensure it exists
  if not os.path.isdir(folder_path):
      print(f"❌ Error: The directory '{folder_path}' does not exist.")
      return []

  # 2. Define the search pattern (looking for .parquet files)
  search_pattern = os.path.join(folder_path, "*.parquet")

  # 3. Find files
  files = glob.glob(search_pattern)

  # 4. Sort alphabetically (ensures shard_001 comes before shard_002)
  files.sort()

  return files

files_in_drive = get_parquet_files(TARGET_DIR)
files_in_local = get_parquet_files(LOCAL_DIR)
print(f"Found {len(files_in_drive)} files in drive.")
print(f"Found {len(files_in_local)} files locally.")

# 1. Get just the filenames (strip the directory path)
filenames_in_drive = {os.path.basename(f) for f in files_in_drive}

# 2. Filter local files that are NOT in the drive set
missing_in_drive = [f for f in files_in_local if os.path.basename(f) not in filenames_in_drive]
print(f"syncing {len(missing_in_drive)} files in drive.")

In [ ]:
def save_part_to_drive(batch_list, part_index,save_path = TARGET_DIR):
    """
    Saves a specific chunk of data as a standalone Parquet file.
    """
    try:
        df = pd.DataFrame(batch_list)
        # Unique filename: part_0.parquet, part_1.parquet, etc.
        filename = f"cc3m_results_shard_{part_index:04d}.parquet"
        drive_path = os.path.join(save_path, filename)

        # Save directly to Drive (small files are safe to write directly)
        df.to_parquet(drive_path, index=False, engine='pyarrow')
        print(f"✓ Saved Part {part_index} to Drive: {len(df)} rows.")
        return True
    except Exception as e:
        print(f"✗ Save Error: {e}")
        return False

In [ ]:
# recommended to run this on a gpu with flash_attn installed
# don't set attn_implemetation if you don't have flash_attn
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    attn_implementation="flash_attention_2"
).to(DEVICE)
model.eval()
model.half()
ip = IndicProcessor(inference=True)

tokenizer_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenization_indictrans.py:   0%|          | 0.00/8.04k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json:   0%|          | 0.00/645k [00:00<?, ?B/s]

dict.TGT.json:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

model.SRC:   0%|          | 0.00/759k [00:00<?, ?B/s]

... (5 outputs left out)

modeling_indictrans.py:   0%|          | 0.00/79.8k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

In [ ]:
inference_dataset = load_dataset("parquet", data_files=missing_in_drive, split="train")
dataloader = DataLoader(
  inference_dataset,
  batch_size=BATCH_SIZE,
  num_workers=4,
  pin_memory=True,
  prefetch_factor=3,
  persistent_workers=True)
translations_to_sync = []

Resolving data files:   0%|          | 0/130 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
executor = ThreadPoolExecutor(max_workers=1)
save_futures = []
saved_counter = len(filenames_in_drive)+1
with torch.inference_mode():
  for batch in tqdm(dataloader,desc="Translating CC3M"):
    ids = batch["id"]
    texts = batch["text"]
    urls = batch["url"]
    itm_scores = batch["itm_score"]

    # 2. Indic Preprocessing (Crucial for script accuracy)
    preprocessed_batch = ip.preprocess_batch(
        texts,
        src_lang=SOURCE_LANGUAGE,
        tgt_lang=TARGET_LANGUAGE
    )

    # 3. Tokenization & Generation
    inputs = tokenizer(
        preprocessed_batch,
        truncation=True,
        padding="longest",
        return_tensors="pt",
        return_attention_mask=True,
    ).to(DEVICE,non_blocking=True)

    generated_tokens = model.generate(
          **inputs,
          use_cache=True,
          min_length=0,
          max_new_tokens=MAX_NEW_TOKENS,
          num_beams=NUM_BEAMS,
          num_return_sequences=1,
          repetition_penalty=REPETITION_PENALTY,
          no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
          early_stopping=EARLY_STOPPING
      )

    # 4. Decoding & Indic Postprocessing
    decoded_batch = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True,clean_up_tokenization_spaces=True)
    final_translations = ip.postprocess_batch(decoded_batch, lang=TARGET_LANGUAGE)

    del inputs,generated_tokens
    torch.cuda.empty_cache()      # free fragmented memory


    for idx, trans in enumerate(final_translations):
      translations_to_sync.append({
              "id": ids[idx],
              "translated_text": trans,
              "text": texts[idx],
              "url":  urls[idx],
              "itm_scores":float(itm_scores[idx])
          })

    # Async save
    if len(translations_to_sync) >= SAVE_THRESHOLD:
        future = executor.submit(save_part_to_drive,
                                list(translations_to_sync), saved_counter)
        save_futures.append(future)
        saved_counter += 1
        translations_to_sync = []

  # Save remaining
  if translations_to_sync:
      future = executor.submit(save_part_to_drive,
                              translations_to_sync, saved_counter)
      save_futures.append(future)

  for future in save_futures:
    try:
        future.result()
    except Exception as e:
        print(f"Save failed: {e}")

executor.shutdown()